# Tutorial 10-3: Density-Based Clustering (DBSCAN)
**Course:** CSEN 140: Machine Learning/Data Mining  
**Instructor:** Dr. David C. Anastasiu

---
## 1. Introduction: Clustering by Density
As we have seen, **k-means** relies on the distance to a centroid and assumes that clusters are globular (spherical). When clusters have complex, non-convex shapes, or when the data contains significant noise, k-means often fails to provide a natural grouping.

**DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) solves this by defining clusters as continuous regions of high density. 

### Key Concepts from Lecture
DBSCAN classifies every point into one of three types based on two parameters: **Eps** (radius) and **MinPts** (density threshold):
1. **Core Point:** Has at least `MinPts` within a radius of `Eps`.
2. **Border Point:** Has fewer than `MinPts` within `Eps`, but is in the neighborhood of a core point.
3. **Noise Point:** Any point that is neither a core point nor a border point.

### The Objective
In this tutorial, we will:
1. Visualize why k-means fails on non-globular data (the 'Moons' dataset).
2. Demonstrate how DBSCAN correctly identifies density-based clusters.
3. Explore the sensitivity of DBSCAN to its hyperparameters.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, DBSCAN
from sklearn.datasets import make_moons

# Step 1: Generate 'Moons' data - two interleaving half-circles
X, y_true = make_moons(n_samples=300, noise=0.05, random_state=42)

plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], s=30, color='gray', alpha=0.5)
plt.title("Non-Globular Data: The Moons Dataset")
plt.xlabel("Feature $x_1$")
plt.ylabel("Feature $x_2$")
plt.show()

## 2. Failure of the Centroid Approach
Below, we apply k-means to this data. Because k-means wants to find convex 'blobs', it is forced to cut through the middle of the moons to find the centroids that minimize the squared distance.

In [ ]:
kmeans = KMeans(n_clusters=2, n_init=10, random_state=42)
km_labels = kmeans.fit_predict(X)

plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], c=km_labels, s=30, cmap='viridis')
plt.title("k-means Attempt\n(Fails to capture the crescent shapes)")
plt.show()

## 3. Success of the Density Approach
Now we apply DBSCAN. Notice that DBSCAN does not require us to specify the number of clusters ($k$). Instead, it follows the 'path' of high density to group the crescents naturally.

In [ ]:
# eps: The maximum distance between two samples for one to be considered as in the neighborhood of the other.
# min_samples: The number of samples in a neighborhood for a point to be considered as a core point.
dbscan = DBSCAN(eps=0.2, min_samples=5)
db_labels = dbscan.fit_predict(X)

plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], c=db_labels, s=30, cmap='plasma')
plt.title("DBSCAN Result (eps=0.2, min_samples=5)\n(Successfully identifies non-globular shapes)")
plt.show()

## 4. Understanding Noise and Hyperparameters
One of DBSCAN's greatest strengths is **Noise Robustness**. Points that do not meet the density requirement and are not near core points are labeled as `-1` (Noise).

However, choosing **Eps** is critical. If Eps is too small, a single cluster might be broken into many small pieces. If Eps is too large, all clusters might merge into one.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5))

eps_values = [0.05, 0.2, 0.5]
for i, e in enumerate(eps_values):
    db = DBSCAN(eps=e, min_samples=5)
    labels = db.fit_predict(X)
    
    # Points with label -1 are noise (plotted in black)
    ax[i].scatter(X[:, 0], X[:, 1], c=labels, s=30, cmap='plasma')
    ax[i].set_title(f"Eps = {e}")
    
    # Count clusters and noise
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)
    ax[i].set_xlabel(f"Clusters: {n_clusters}, Noise: {n_noise}")

plt.suptitle("Sensitivity to Eps Parameter")
plt.show()

## 5. Identifying Core, Border, and Noise Points
We can inspect the internal properties of the DBSCAN object to see which points are 'Core Points' (those that drive the creation of clusters).

In [ ]:
db = DBSCAN(eps=0.2, min_samples=5)
labels = db.fit_predict(X)

# core_sample_indices_ gives us the core points
core_mask = np.zeros_like(labels, dtype=bool)
core_mask[db.core_sample_indices_] = True

plt.figure(figsize=(8, 6))
# Plot Core Points
plt.scatter(X[core_mask, 0], X[core_mask, 1], c='green', marker='o', label='Core Points')
# Plot Border Points (Assigned to cluster but not core)
plt.scatter(X[~core_mask & (labels != -1), 0], X[~core_mask & (labels != -1), 1], c='blue', marker='s', label='Border Points')
# Plot Noise
plt.scatter(X[labels == -1, 0], X[labels == -1, 1], c='black', marker='x', label='Noise')

plt.title("Visualizing Point Types in DBSCAN")
plt.legend()
plt.show()

## Summary

### 1. Strengths of DBSCAN
* **Shape Agnostic:** Can find clusters of any arbitrary shape.
* **No Prior K:** You do not need to know the number of clusters in advance.
* **Robustness:** Excellent at filtering out outliers and noise.

### 2. Weaknesses of DBSCAN
* **Varying Densities:** If different clusters have vastly different densities, a single Eps value will fail to find all of them.
* **High Dimensions:** In high-dimensional space, data becomes sparse, making density difficult to define (the 'Curse of Dimensionality').

### 3. Choosing Parameters
As discussed in **Slide 41**, a common heuristic for choosing Eps is to plot the distance to the $k^{th}$ nearest neighbor for every point. The 'elbow' in that plot usually suggests an appropriate Eps value.